# 🚂 RailPulse: AI-Powered Railway Risk & Resource Allocation
**Hackathon Presentation Notebook**

This notebook walks through the data generation, feature engineering, and model training process for RailPulse. 
Our model predicts coach-level overcrowding, ticketless travel, and station congestion using existing booking data.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the synthetic training data
df = pd.read_csv('../data/train_data.csv')
print(f"Dataset loaded: {len(df):,} records")
df.head()


## 📊 1. Exploratory Data Analysis
Let's look at the distribution of occupancy and the impact of festivals.


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['occupancy_ratio'], bins=50, kde=True, color='purple')
plt.title('Distribution of Coach Occupancy Ratio')
plt.axvline(1.0, color='red', linestyle='--', label='100% Capacity')
plt.legend()
plt.show()


In [ ]:
# Impact of festivals on overcrowding
festival_impact = df.groupby('day_type')['overcrowding_label'].mean() * 100
print("Overcrowding Probability by Day Type:")
print(festival_impact)


## ⚙️ 2. Feature Engineering & Model Training
We use XGBoost classifiers to predict two separate risks:
1. **Overcrowding Risk** (Probability of >100% capacity + standing passengers)
2. **Ticketless Travel Risk** (Probability of unauthorized passengers)


In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Features for Overcrowding Model
features = ['is_holiday', 'is_weekend', 'occupancy_ratio', 'historical_avg_occupancy', 'route_popularity']
X = df[features]
y = df['overcrowding_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBClassifier(eval_metric='logloss')
model.fit(X_train, y_train)

preds = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

print(f"AUC Score: {roc_auc_score(y_test, proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, preds))


## 📈 3. Feature Importance
What drives the AI's decisions? Let's check what features the model prioritizes.


In [ ]:
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
plt.figure(figsize=(8, 4))
sns.barplot(x=importances.values, y=importances.index, palette='viridis')
plt.title('Top Risk Factors for Overcrowding')
plt.xlabel('XGBoost Feature Importance')
plt.show()
